# Análisis V2 - Support Vector Machines: SVR y SVC

## 1. Contexto y Diagnóstico del Fallo en V1
En la versión V1, el regresor SVR arrojó un $R^2 = -0.0080$ con predicciones planas en cero debido a que el target continuo no fue escalado y la penalización $C=10$ era microscópica frente a quintales en cientos de miles. En esta versión V2 se aplicó la transformación logarítmica bilog $\ln(1 + x)$ combinada con estandarización, y se evaluó el clasificador SVC sobre el dataset balanceado V3.

## 2. Metodología
* **SVR V2:** `0901_Produccion_Arroz_V2_Regresion.xlsx` (1,373 filas). Estandarización de $X_{\text{log}}$ y optimización vía `GridSearchCV(KFold=5)` sobre $C \in [1, 500]$, $\epsilon \in [0.01, 0.2]$ y $\gamma$.
* **SVC V2:** `0901_Produccion_Arroz_V3_Clasificacion.xlsx` (280 filas), kernel RBF, proyección 2D mediante PCA para inspección geométrica de vectores de soporte.

## 3. Resultados y Comparación Histórica

### SVR en Regresión (Quintales Reales):
* **Hiperparámetros V2:** `kernel = 'rbf'`, $C = 100$, $\epsilon = 0.01$, $\gamma = 0.1$.

| Métrica | SVR V1 (Sin Escalar Target) | SVR V2 (Con $\ln(1+x)$ + Estandarización) |
|---|:---:|:---:|
| **$R^2$ Score** | -0.0080 | **0.2207** |
| **MAE** | 4,226.07 qq | **1,485.89 qq** |
| **RMSE** | 40,355.34 qq | **8,192.08 qq** |
| **MAPE** | > 300% | **55.58%** |

### SVC en Clasificación (Test 42 vs 42):
* **Hiperparámetros V2:** `kernel = 'rbf'`, $C = 50$, $\gamma = \text{'scale'}$.

| Métrica | SVC V1 (Desbalanceado + Balanced) | SVC V2 (Dataset Balanceado 50/50) |
|---|:---:|:---:|
| **Accuracy** | 83.05% | **59.52%** |
| **Precisión (Inv)** | 26.67% | **72.22%** |
| **Recall (Inv)** | 38.10% (16/42) | **30.95% (13/42)** |
| **F1-Score (Inv)** | 0.3137 | **0.4333** |
| **ROC-AUC** | 0.6840 | **0.7574** |

## 4. Interpretación Técnica
1. **Recuperación del SVR:** Al normalizar las magnitudes de penalización, el SVR logró converger y reducir el MAE de 4,226 qq a 1,485.89 qq. Sin embargo, el tubo $\epsilon$ tiende a curvarse hacia arriba en superficies superiores a 1,500 Ha, limitando su $R^2$ al 22.07%.
2. **Geometría de Soporte en SVC:** La proyección bidimensional PCA demuestra que las muestras de verano e invierno están altamente concentradas en el origen. El hiperplano RBF define una frontera conservadora alrededor de los vectores de soporte, logrando alta precisión (72.22%) a expensas de un Recall modesto (30.95%).

## 5. Visualización
![Diagnóstico SVR V2](../03_imagenes/39_svm_reg_v2.png)
![Diagnóstico SVC V2](../03_imagenes/40_svm_clf_v2.png)

## 6. Conclusión
Aunque las Máquinas de Vectores de Soporte fueron recuperadas de sus fallos iniciales, su complejidad computacional $\mathcal{O}(n^3)$ y sus métricas inferiores respecto a LightGBM ($R^2 = 0.768$) y CatBoost ($F_1 = 0.809$) aconsejan descartarlas de la arquitectura de producción web.